# Benchmark Exploration

This notebook provides a quick smoke test and visualization for the models and methods registered in the `xai_bench` package.

It loads a sample image, runs attribution methods, and visualizes the results.

In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, os.path.abspath('.'))

from xai_bench.registry import MODELS, METHODS, DATASETS
from xai_bench.config import RunConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Data
Load our demo dataset consisting of synthetic images and bounding boxes.

In [ ]:
dataset_fn = DATASETS.get('demo')
dataset = dataset_fn(root='data/demo/images', bbox_dir='data/demo/bboxes', labels_file='data/demo/labels.txt')
print(f"Loaded dataset with {len(dataset)} samples.")

# Get first sample
img_tensor, target_label, metadata = dataset[0]
print(f"Target class: {target_label}")
print(f"Metadata: {metadata}")

## 2. Load Model & Setup Methods
We'll load a standard ResNet50 as a quick smoke test.

In [ ]:
model_name = 'resnet50'
model_wrapper = MODELS.get(model_name)()
model_wrapper.model.to(device)
model_wrapper.model.eval()
print(f"{model_name} loaded.")

methods_to_test = ['saliency', 'grad_cam']
attributions = {}

batch_img = img_tensor.unsqueeze(0).to(device)

for method_name in methods_to_test:
    method_fn = METHODS.get(method_name)
    method = method_fn(model_wrapper=model_wrapper, model=model_wrapper.model)
    
    attr = method(batch_img, target=target_label)
    attributions[method_name] = attr[0].cpu().detach().numpy()

## 3. Visualize Attributions

In [ ]:
def unnormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

fig, axes = plt.subplots(1, len(attributions) + 1, figsize=(15, 5))

# Original image
img_display = unnormalize(img_tensor).permute(1, 2, 0).numpy()
axes[0].imshow(img_display.clip(0, 1))
axes[0].set_title("Original Image")
axes[0].axis('off')

# Attributions
for i, (method_name, attr) in enumerate(attributions.items(), 1):
    axes[i].imshow(attr, cmap='jet')
    axes[i].set_title(method_name)
    axes[i].axis('off')

plt.tight_layout()
plt.show()